# 🚗 UA-DETRAC Trafik Veri Seti ile YOLO11 Araç Tespiti (Uçtan Uca Pipeline)
### Bitirme Projesi - Gerçek Trafik Kameraları ile Yüksek Başarımlı Model Eğitimi

Bu notebook, Google Drive'ınızdaki **`DETRAC-Images.zip`** arşivini otomatik olarak bulur, etiketleri işler, **YOLO11** modelini T4 GPU ile eğitir ve sonuçları paketler.

👉 **Tüm adımları yukarıdan aşağıya sırayla (1'den 7'ye) çalıştırmanız yeterlidir.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## Adım 1: Donanım Kontrolü ve Kütüphane Kurulumu
GPU hızlandırıcısının (T4) devrede olduğunu doğrulayıp Ultralytics (YOLO11) kütüphanesini kuruyoruz.


In [ ]:
import os
import torch

if torch.cuda.is_available():
    print(f'✅ Harika! T4 GPU devrede: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL! Lütfen üst menüden:')
    print('Çalışma zamanı (Runtime) -> Çalışma zamanı türünü değiştir (Change runtime type) -> T4 GPU seçip Kaydet deyin.')

!pip install -q ultralytics
print('✅ Ultralytics ve PyTorch ortamı hazır!')


## Adım 2: Proje Deposunu ve XML Etiketlerini Çekme
GitHub üzerindeki repomuzu ve içerisinde hazır bulunan XML koordinat etiketlerini çekiyoruz.


In [ ]:
import os

repo_dir = '/content/federated-traffic-learning'
if not os.path.exists(repo_dir):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git /content/federated-traffic-learning
    %cd /content/federated-traffic-learning
else:
    %cd /content/federated-traffic-learning
    !git pull

print('✅ Proje dizini ve XML etiket paketi hazır:', os.getcwd())


## Adım 3: Google Drive'daki DETRAC-Images.zip'i Bulma ve YOLO Formatına Dönüştürme
Drive'a attığınız **`DETRAC-Images.zip`** dosyasını büyük/küçük harf duyarsız olarak bulur, repodaki XML etiketleriyle eşleştirip doğrudan YOLO formatına dönüştürür.


In [ ]:
import os
import glob
import zipfile
import random
import shutil
import xml.etree.ElementTree as ET
from google.colab import drive

# 1. Google Drive'ı bağla
if not os.path.exists('/content/drive/MyDrive'):
    print("🔌 Google Drive bağlanıyor...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive zaten bağlı.")

# 2. Eski/hatalı kalıntıları temizle (temiz bir başlangıç için)
if os.path.exists('/content/ua_detrac_dataset'):
    shutil.rmtree('/content/ua_detrac_dataset', ignore_errors=True)
if os.path.exists('/content/ua_detrac_yolo.zip'):
    try:
        os.remove('/content/ua_detrac_yolo.zip')
    except:
        pass

# 3. Google Drive'da 9.16 GB'lık DETRAC-Images.zip dosyasını ara
print("🔍 Google Drive içerisinde DETRAC-Images.zip (9.16 GB) aranıyor...")
drive_root = '/content/drive/MyDrive'
found_zips = []

for root, dirs, files in os.walk(drive_root):
    if '.trash' in root.lower():
        continue
    for f in files:
        if f.lower().endswith('.zip') and ('detrac' in f.lower() or 'images' in f.lower()):
            full_p = os.path.join(root, f)
            try:
                sz = os.path.getsize(full_p)
                found_zips.append((full_p, sz))
            except:
                pass

# Dosya boyutuna göre sırala (9.16 GB olan DETRAC-Images.zip en başta gelsin)
found_zips.sort(key=lambda x: x[1], reverse=True)

if not found_zips:
    all_drive_zips = []
    for root, dirs, files in os.walk(drive_root):
        if '.trash' in root.lower():
            continue
        for f in files:
            if f.lower().endswith('.zip'):
                all_drive_zips.append((os.path.join(root, f), os.path.getsize(os.path.join(root, f))))
    print("❌ DETRAC-Images.zip Google Drive'da bulunamadı!")
    if all_drive_zips:
        print("Drive'ınızdaki mevcut zip dosyaları:")
        for p, sz in all_drive_zips:
            print(f"  - {p} ({sz / (1024*1024):.1f} MB)")
    raise FileNotFoundError("Google Drive içerisinde DETRAC-Images.zip bulunamadı! Dosyanın Drive hesabınızda olduğundan emin olun.")

images_zip_path, zip_size = found_zips[0]
print(f"✅ Doğru veri seti arşivi bulundu: {images_zip_path} ({zip_size / (1024**3):.2f} GB)")

# 4. XML Etiket Arşivini Aç
xml_zip_path = '/content/federated-traffic-learning/detrac_xml_annotations.zip'
annot_dir = '/content/DETRAC_annotations'
if not os.path.exists(annot_dir):
    print("📦 XML etiketleri açılıyor...")
    with zipfile.ZipFile(xml_zip_path, 'r') as xz:
        xz.extractall(annot_dir)
    print("✅ 93 adet XML video etiketi hazır!")
else:
    print("✅ XML etiketleri hazır.")

# 5. Çıktı Klasörlerini Hazırla
out_dataset_dir = '/content/ua_detrac_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(out_dataset_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(out_dataset_dir, 'labels', split), exist_ok=True)

CLASS_MAP = {'car': 0, 'van': 0, 'bus': 2, 'others': 3}
IMG_W, IMG_H = 960.0, 540.0

def parse_xml_file(xml_p):
    tree = ET.parse(xml_p)
    root = tree.getroot()
    seq = root.attrib.get('name', os.path.splitext(os.path.basename(xml_p))[0]).strip()
    frames = {}
    for fe in root.iter('frame'):
        n = fe.attrib.get('num')
        if not n:
            continue
        fnum = int(n)
        boxes = []
        tl = fe.find('target_list')
        if tl is not None:
            for tgt in tl.findall('target'):
                bx = tgt.find('box')
                att = tgt.find('attribute')
                if bx is None or att is None:
                    continue
                vt = att.attrib.get('vehicle_type', 'car').lower()
                if vt not in CLASS_MAP:
                    continue
                cid = CLASS_MAP[vt]
                try:
                    l = float(bx.attrib.get('left', 0))
                    t = float(bx.attrib.get('top', 0))
                    w = float(bx.attrib.get('width', 0))
                    h = float(bx.attrib.get('height', 0))
                except:
                    continue
                if w < 5 or h < 5:
                    continue
                xc = max(0.0, min(1.0, (l + w / 2.0) / IMG_W))
                yc = max(0.0, min(1.0, (t + h / 2.0) / IMG_H))
                wn = max(0.001, min(1.0, w / IMG_W))
                hn = max(0.001, min(1.0, h / IMG_H))
                boxes.append((cid, xc, yc, wn, hn))
        frames[fnum] = boxes
    return seq, frames

train_xml_files = sorted(glob.glob(f'{annot_dir}/train/*.xml'))
test_xml_files = sorted(glob.glob(f'{annot_dir}/test/*.xml'))

random.seed(42)
random.shuffle(train_xml_files)
val_count = 9
val_xmls = train_xml_files[:val_count]
train_xmls = train_xml_files[val_count:]

print(f"🎬 Video Dizileri: Train={len(train_xmls)}, Val={len(val_xmls)}, Test={len(test_xml_files)}")
print("🚀 9.16 GB'lık arşivden en kaliteli sahneler örneklenip etiketleniyor (1-2 dakika sürer)...")

with zipfile.ZipFile(images_zip_path, 'r') as zf:
    zip_keys = set(zf.namelist())

    def extract_split(xml_list, split_name, step):
        img_dir = os.path.join(out_dataset_dir, 'images', split_name)
        lbl_dir = os.path.join(out_dataset_dir, 'labels', split_name)
        extracted_imgs = 0
        extracted_boxes = 0
        
        for i, xp in enumerate(xml_list):
            seq_name, f_dict = parse_xml_file(xp)
            for fnum in sorted(f_dict.keys())[::step]:
                boxes = f_dict[fnum]
                if not boxes:
                    continue
                key = f'DETRAC-Images/{seq_name}/img{fnum:05d}.jpg'
                if key not in zip_keys:
                    key = f'{seq_name}/img{fnum:05d}.jpg'
                    if key not in zip_keys:
                        continue
                fname = f'{seq_name}_img{fnum:05d}'
                with open(os.path.join(img_dir, f'{fname}.jpg'), 'wb') as img_f:
                    img_f.write(zf.read(key))
                with open(os.path.join(lbl_dir, f'{fname}.txt'), 'w', encoding='utf-8') as lbl_f:
                    for b in boxes:
                        lbl_f.write(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n')
                extracted_imgs += 1
                extracted_boxes += len(boxes)
                
            if (i + 1) % 10 == 0 or (i + 1) == len(xml_list):
                print(f"   [{split_name.upper()}] {i+1}/{len(xml_list)} video dizisi işlendi ({extracted_imgs} görsel, {extracted_boxes} araç)...")
                
        return extracted_imgs, extracted_boxes

    train_imgs, train_boxes = extract_split(train_xmls, 'train', step=20)
    val_imgs, val_boxes = extract_split(val_xmls, 'val', step=20)
    test_imgs, test_boxes = extract_split(test_xml_files, 'test', step=30)

print('\n' + '=' * 60)
print('✅ ADIM 3 BAŞARIYLA TAMAMLANDI!')
print(f' • Toplam Oluşturulan Görsel: {train_imgs + val_imgs + test_imgs:,} adet')
print(f' • Toplam Etiketlenen Araç:   {train_boxes + val_boxes + test_boxes:,} adet')
print(f' • Hedef Dizin: {out_dataset_dir}')
print('=' * 60)



## Adım 4: Veri Seti Detaylı Analiz Raporu ve data.yaml Yapılandırması 📊
Hazırlanan veri setindeki tüm araç sınıflarını (`car`, `motorcycle`, `bus`, `truck`), etiketli araç sayılarını ve oranlarını ekrana döker, `data.yaml` dosyasını oluşturur.


In [ ]:
import os
import glob
from collections import Counter

dataset_root = '/content/ua_detrac_dataset'
data_yaml_path = f'{dataset_root}/data.yaml'

# 1. data.yaml Yapılandırmasını Oluştur
yaml_text = f"""# Vehicle Detection YOLO Dataset Configuration (UA-DETRAC Benchmark)
path: {dataset_root}
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""

with open(data_yaml_path, 'w', encoding='utf-8') as yf:
    yf.write(yaml_text.strip() + '\n')

# 2. Detaylı Sınıf ve Kutu Dağılımını Analiz Et
class_names_map = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}
print('=' * 68)
print('         📊 YENİ TRAFİK VERİ SETİ DETAYLI ANALİZ RAPORU')
print('=' * 68)
print(f'📁 Veri Seti Dizin Yolu: {dataset_root}\n')

total_overall_boxes = 0
overall_class_counts = Counter()

for split_name in ['train', 'val', 'test']:
    img_dir = os.path.join(dataset_root, 'images', split_name)
    lbl_dir = os.path.join(dataset_root, 'labels', split_name)
    
    imgs = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    lbls = glob.glob(f'{lbl_dir}/*.txt')
    
    class_counts = Counter()
    split_boxes = 0
    for lf in lbls:
        try:
            with open(lf, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls_id = int(float(parts[0]))
                        class_counts[cls_id] += 1
                        split_boxes += 1
        except:
            pass
            
    overall_class_counts.update(class_counts)
    total_overall_boxes += split_boxes
    
    print(f'📌 [{split_name.upper()} KÜMESİ]')
    print(f'   • Toplam Görsel Sayısı: {len(imgs):,}')
    print(f'   • Toplam Araç Kutusu (Bounding Box): {split_boxes:,}')
    print('   • Sınıf Dağılımı:')
    for cid in sorted(class_names_map.keys()):
        if cid in class_counts:
            cname = class_names_map[cid]
            pct = (class_counts[cid] / split_boxes * 100) if split_boxes > 0 else 0.0
            print(f'      - ID {cid:02d} ({cname:<12}): {class_counts[cid]:>7,} adet  (%{pct:>5.1f})')
    print('-' * 68)

print('\n🎯 [GENEL TOPLAM]')
print(f'   • Toplam Etiketlenen Araç Sayısı: {total_overall_boxes:,}')
for cid in sorted(overall_class_counts.keys()):
    cname = class_names_map.get(cid, f'class_{cid}')
    pct = (overall_class_counts[cid] / total_overall_boxes * 100) if total_overall_boxes > 0 else 0.0
    print(f'   • Sınıf {cid} ({cname:<12}): {overall_class_counts[cid]:>8,} adet  (%{pct:>5.1f})')
print('=' * 68)
print(f'✅ data.yaml hazır: {data_yaml_path}')



## Adım 5: YOLO11 Modelini Eğitme (Training) 🚀
Hazırlanan veri seti üzerinde **YOLO11 Nano** modelini T4 GPU ile eğitiyoruz.


In [ ]:
import os
import torch
from ultralytics import YOLO

device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'🚀 Model Eğitimi Başlatılıyor... Donanım: {device_arg} (GPU: {torch.cuda.is_available()})')

# YOLO11 Nano modelini yüklüyoruz
model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/ua_detrac_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device_arg,
    project='runs/train',
    name='yolo11n_uad_exp',
    patience=10,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


## Adım 6: Model Başarım Testi ve Gerçek Fotoğraf Kıyaslaması 📸
Bağımsız test kümesi üzerinde kesinlik (mAP) değerlerini ölçer ve özel test fotoğrafı üzerinde eğitim öncesi vs. sonrası tespit farkını yan yana görselleştirir.


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# 1. En iyi model ağırlığını belirle
trained_weights = 'runs/train/yolo11n_uad_exp/weights/best.pt'
if not os.path.exists(trained_weights):
    cand = glob.glob('runs/**/best.pt', recursive=True)
    if cand:
        trained_weights = cand[-1]

print(f"🔍 Değerlendirilen En İyi Model: {trained_weights}")
trained_model = YOLO(trained_weights)

# 2. Test Kümesi Değerlendirmesi
metrics = trained_model.val(data='/content/ua_detrac_dataset/data.yaml')
print('\n' + '=' * 50)
print('      MODEL BAŞARI METRİKLERİ (UA-DETRAC)')
print('=' * 50)
print(f"mAP@50:     {metrics.box.map50:.4f} (%{metrics.box.map50 * 100:.2f})")
print(f"mAP@50-95:  {metrics.box.map:.4f} (%{metrics.box.map * 100:.2f})")
print(f"Precision:  {metrics.box.mp:.4f} (%{metrics.box.mp * 100:.2f})")
print(f"Recall:     {metrics.box.mr:.4f} (%{metrics.box.mr * 100:.2f})")
print('=' * 50)

# 3. Gerçek Trafik Test Fotoğrafı Üzerinde Kıyaslama
custom_photo = '/content/federated-traffic-learning/test_samples/user_test_photo.jpg'
if not os.path.exists(custom_photo):
    custom_photo = 'test_samples/user_test_photo.jpg'

if os.path.exists(custom_photo) and os.path.exists(trained_weights):
    print('\n📸 Özel test fotoğrafı kıyaslaması yapılıyor...')
    base_model = YOLO('yolo11n.pt')
    base_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='base_pred', exist_ok=True)
    
    trained_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='trained_pred', exist_ok=True)
    
    base_imgs = glob.glob('runs/detect/base_pred/*.*')
    trained_imgs = glob.glob('runs/detect/trained_pred/*.*')
    
    if base_imgs and trained_imgs:
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(Image.open(base_imgs[0]))
        axes[0].set_title('Eğitim Öncesi: Base YOLO11n (Pretrained)', fontsize=15)
        axes[0].axis('off')
        
        axes[1].imshow(Image.open(trained_imgs[0]))
        axes[1].set_title('Eğitim Sonrası: UA-DETRAC ile Eğitilmiş Model', fontsize=15)
        axes[1].axis('off')
        
        plt.tight_layout()
        out_comp = 'runs/train/yolo11n_uad_exp/custom_test_comparison.jpg'
        plt.savefig(out_comp, bbox_inches='tight', dpi=150)
        print(f'✅ Karşılaştırma görseli kaydedildi: {out_comp}')
        plt.show()



## Adım 7: Sonuçları Paketleme ve Otomatik İndirme 📦
Eğitilen `best.pt` model ağırlıklarını, grafiklerini ve karşılaştırma görselini tek bir `yolo11_results.zip` dosyası haline getirip bilgisayarınıza indirir.


In [ ]:
import os
import glob
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11n_uad_exp'
if not os.path.exists(exp_dir):
    cand = glob.glob('runs/train/*')
    if cand:
        exp_dir = cand[-1]

best_weights = f'{exp_dir}/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ Eğitilmiş en başarılı model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', exp_dir)
files.download('yolo11_results.zip')
print('\n🎉 TEBRİKLER! Tüm eğitim ve test işlemleri başarıyla tamamlandı!')
print('👉 İnen "yolo11_results.zip" dosyasını masaüstündeki proje klasörünüze atıp asistanınıza bildirebilirsiniz.')

